# Window Functions & Incrementalisation

Considering this query joining tables from bronze

In [0]:
CREATE OR REPLACE TEMP VIEW language_enriched AS 
WITH base AS (
    SELECT

        CAST(rla.attempt_id AS BIGINT)                    AS attempt_id
        , CAST(rla.learner_id AS INT)                       AS learner_id        
        , rla.course_id
        , CAST(rla.lesson_number AS INT)                    AS lesson_number
        , CAST(rla.attempted_at AS TIMESTAMP)               AS attempted_at
        , CAST(rla.attempted_at AS DATE)                    AS attempt_date    
        , CAST(rla.score AS INT)                            AS score
        , CASE WHEN rla.minutes_spent >= 0
            THEN CAST(rla.minutes_spent AS INT)
        END                                               AS minutes_spent     
                                                                                
        , CAST(rla.completed AS BOOLEAN)                    AS completed
        , rla.device

        , rl.learner_id IS NOT NULL                         AS learner_found    
        , COALESCE(rl.first_name || ' ' || rl.last_name,
                'Unknown')                               AS learner_name      
        , COALESCE(rl.country, 'Unknown')                   AS learner_country   
        , COALESCE(rl.native_language, 'Unknown')           AS native_language   
        , COALESCE(rl.subscription_tier, 'Unknown')         AS subscription_tier 
        , CAST(rl.signup_date AS DATE)                      AS signup_date       

        , COALESCE(rc.language, 'Unknown')                  AS course_language  
        , COALESCE(rc.level, 'Unknown')                     AS course_level     
        , COALESCE(rc.course_name, 'Unknown')               AS course_name      
        , CAST(rc.total_lessons AS INT)                     AS course_total_lessons 

    FROM language_app.bronze.raw_lesson_attempts rla

    LEFT JOIN language_app.source.raw_learners rl
        ON rla.learner_id = rl.learner_id
    LEFT JOIN language_app.bronze.raw_courses rc
        ON rla.course_id = rc.course_id
)
SELECT * FROM base
;

It is incrementalisable

In [0]:
EXPLAIN CREATE MATERIALIZED VIEW language_app.silver.lesson_attempts_enriched
SELECT * 
FROM language_enriched
;

But adding a global window function, it is no longer incrementalisable. Assume you want to rank scorers high to low:

We get an error message (an error in the sense it is not incrementalisable):
```
WINDOW_WITHOUT_PARTITION_BY: Window specifications without PARTITION BY are not incrementalisable. Consider adding PARTITION BY clauses to the window specifications.
```

In [0]:
EXPLAIN CREATE MATERIALIZED VIEW language_app.silver.test
select
  *,
  rank(score) over (order by score desc)
from
  language_enriched

To fix this we need to add `PARTITION BY`

Why? So that only the partition where a change happened needs to be updated, NOT the whole table. In this example we rank scores per person, and even if a new score lands somewhere in the middle, only that partition needs to be updated to reflect the new rankings

In [0]:
EXPLAIN CREATE MATERIALIZED VIEW language_app.silver.test
select
  *,
  rank(score) over (partition by learner_id order by score desc)
from
  language_enriched

The lesson: A non-incrementalisable query does not always mean you have to remove the offending pattern. Sometimes you can rewrite it to achieve the same goal in an incrementalisable way. Always run EXPLAIN before building a materialised view.